# quant-kit: Heavy Benchmarking
Run this notebook on Kaggle using a free **T4 GPU x2** runtime to evaluate your GGUF models on heavy benchmarks like HellaSwag or MMLU.

In [ ]:
!pip install -q lm-evaluation-harness llama-cpp-python huggingface-hub

In [ ]:
import os
import subprocess
import time

# 1. Upload your GGUF file to a Kaggle Dataset and replace the path here:
MODEL_PATH = "/kaggle/input/your-model-dataset/your-model-Q4_K_M.gguf"

print("Starting llama.cpp server...")
server_process = subprocess.Popen([
    "python", "-m", "llama_cpp.server",
    "--model", MODEL_PATH,
    "--n_gpu_layers", "-1", # Offload entirely to T4 GPU
    "--port", "8000"
])

time.sleep(15) # Wait for server to load the model

In [ ]:
# 2. Run the benchmarks
TASKS = "hellaswag,arc_challenge"

!lm_eval --model local-completions \
    --model_args model=gguf-model,base_url=http://localhost:8000/v1/completions,num_concurrent=4 \
    --tasks {TASKS} \
    --output_path ./results

In [ ]:
# 3. View the results
import json
from pathlib import Path

results_dir = Path("./results")
if results_dir.exists():
    for result_file in results_dir.glob("**/*.json"):
        with open(result_file) as f:
            data = json.load(f)
            print(f"\nResults for {result_file.name}:")
            for task, metrics in data.get('results', {}).items():
                print(f"{task}:")
                for metric, value in metrics.items():
                    if isinstance(value, float):
                        print(f"  {metric}: {value:.4f}")
